<h1>Seleccion de modelo por AUC macro de VALIDACION (VinDr, exp01-exp09 / C1-C9 en la tesis)</h1>
<p>Este notebook reconstruye el conjunto de VALIDACION de VinDr (no el test de 4000)
usando unicamente las funciones reales del repositorio, verifica que arquitectura
corresponde a cada checkpoint, hace un sanity check obligatorio sobre exp08 (condicion C8
en la tesis, modelo definitivo) y, solo si ese sanity check pasa, rankea los checkpoints
sobrevivientes por AUC macro one vs rest de la cabeza BI-RADS sobre ese val de 2400
muestras.</p>
<p>No se recalcula nada sobre el test. Todas las afirmaciones del enunciado se tratan
como hipotesis a confirmar contra el codigo y los archivos, no como hechos dados.</p>
<p>El resultado numerico definitivo queda persistido en <code>resultados_ranking.json</code>,
en esta misma carpeta. Las salidas de celda de este notebook se limpian antes de
guardarlo; sirven solo para auditar el proceso durante la ejecucion.</p>

In [5]:
## Localizacion de la raiz del repo de forma relativa al notebook.
## No se escribe ninguna ruta absoluta a mano: se calcula subiendo dos
## niveles desde la carpeta de este notebook (experiments/seleccion_val/).
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
SRC_DIR = REPO_ROOT / "src"
CACHE_DIR = NOTEBOOK_DIR / "_cache"
OUTPUTS_DIR = NOTEBOOK_DIR / "_outputs"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

## src/ se agrega antes que la raiz del repo, igual que en train_runner.py,
## porque los modulos de src/ se importan entre si sin prefijo de paquete
sys.path.insert(0, str(SRC_DIR))
sys.path.insert(0, str(REPO_ROOT))

print("directorio del notebook:", NOTEBOOK_DIR)
print("raiz del repo:", REPO_ROOT)
print("directorio src:", SRC_DIR)

directorio del notebook: /home/gtrujillod/Tesis/experiments/seleccion_val
raiz del repo: /home/gtrujillod/Tesis
directorio src: /home/gtrujillod/Tesis/src


In [9]:
## Importaciones de los modulos reales del repo (src/) y de librerias externas.
## Se usan unicamente los modulos de este repositorio para reconstruir el split,
## la arquitectura y las metricas; ningun snapshot externo se usa en este notebook.
import json
import time
import hashlib
from datetime import datetime, timezone

import numpy as np
import torch
from torch.utils.data import DataLoader

from data_loading import load_vindr_records, MammoCLIPTransform, MammoDataset
from train import separate_test_set_vindr, split_train_val, TrainingConfig
from models import MammoVLM
from medical_metrics import MedicalMetricsReport

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda:0


<h2>Consulta 1a-1c. Reconstruccion del conjunto de VALIDACION</h2>
<p>Se usa la funcion real <code>src/train.py:split_train_val</code>, llamada despues de
<code>src/train.py:separate_test_set_vindr</code> (que separa el test oficial de VinDr
antes de tocar el resto). Semilla 42, estratificado por BI-RADS, val_fraction 0.15.</p>
<p>Nota critica de reproducibilidad: <code>separate_test_set_vindr</code> compara rutas de
imagen con <code>isin()</code> contra el CSV de test guardado en disco. Ese CSV guarda
rutas ABSOLUTAS, por lo que <code>load_vindr_records</code> debe llamarse con una raiz de
dataset tambien absoluta; con una raiz relativa la comparacion de rutas falla en
silencio y el test set no se excluye correctamente.</p>

In [10]:
## Reconstruccion del split train/val/test de VinDr con las funciones reales
## del repo. dataset_root y test_set_dir se pasan como rutas absolutas (ver nota
## de reproducibilidad arriba) para que separate_test_set_vindr excluya bien el test.
DATASET_ROOT = str(REPO_ROOT / "data" / "vindr-mammo")
TEST_SET_DIR = REPO_ROOT / "outputs" / "test_sets"
TEST_SET_PATH = TEST_SET_DIR / "test_set_vindr.csv"

records = load_vindr_records(DATASET_ROOT)
n_total_records = len(records)

split_config = TrainingConfig(
    test_set_dir=str(TEST_SET_DIR),
    use_official_split=True,
    val_fraction=0.15,
    random_seed=42,
)

train_val_df, test_df = separate_test_set_vindr(records, split_config, str(TEST_SET_PATH))
train_df, val_df = split_train_val(train_val_df, val_fraction=0.15, seed=42)

print("funcion de split usada: src/train.py:split_train_val")
print("funcion de reserva de test usada: src/train.py:separate_test_set_vindr")
print("loader de registros usado: src/data_loading.py:load_vindr_records")
print("semilla:", split_config.random_seed, "val_fraction:", split_config.val_fraction)
print("registros totales de VinDr cargados:", n_total_records)
print("test reservado, split oficial de VinDr:", len(test_df))
print("train + val antes del split interno:", len(train_val_df))
print("train:", len(train_df))
print("val:", len(val_df))
print("val por clase BI-RADS:")
print(val_df["birads"].value_counts().sort_index())

funcion de split usada: src/train.py:split_train_val
funcion de reserva de test usada: src/train.py:separate_test_set_vindr
loader de registros usado: src/data_loading.py:load_vindr_records
semilla: 42 val_fraction: 0.15
registros totales de VinDr cargados: 20000
test reservado, split oficial de VinDr: 4000
train + val antes del split interno: 16000
train: 13600
val: 2400
val por clase BI-RADS:
birads
1    1609
2     561
3     112
4      91
5      27
Name: count, dtype: int64


<h2>Preprocesamiento de validacion</h2>
<p>Se usa <code>src/data_loading.py:MammoCLIPTransform</code> sin augmentacion (igual que
en <code>src/train.py</code> para el val_loader de entrenamiento): CLAHE, resize a
1520x912 y normalizacion ImageNet.</p>

In [ ]:
## DataLoader de validacion, identico en preprocesamiento al usado durante
## el entrenamiento de exp06-exp09 (ver src/train.py, construccion de val_loader)
val_transform = MammoCLIPTransform(height=1520, width=912, augment=False, use_clahe=True)
val_ds = MammoDataset(val_df, val_transform, augment=False)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=8, pin_memory=True)

print("muestras en el dataloader de validacion:", len(val_ds))

<h2>Consulta 2a. Inventario fisico de checkpoints (exp01 a exp09)</h2>
<p>Se listan ruta, existencia, tamanio y SHA-256 de cada <code>model.pt</code> en
<code>outputs/experiments/</code>, sin cargar todavia ningun modelo.</p>

In [ ]:
## Inventario fisico de checkpoints. Se usa la ruta model.pt de cada
## experimento en outputs/experiments/, que es donde vive el checkpoint final
## registrado por experimento (distinto de outputs/checkpoints/, que guarda los
## checkpoints intermedios por epoca durante el entrenamiento).
EXPERIMENTS_DIR = REPO_ROOT / "outputs" / "experiments"

checkpoint_paths = {
    "exp01": EXPERIMENTS_DIR / "exp01_baseline_encoder_congelado" / "model.pt",
    "exp02": EXPERIMENTS_DIR / "exp02_encoder_descongelado_3bloques" / "model.pt",
    "exp03": EXPERIMENTS_DIR / "exp03_focal_loss_encoder_congelado" / "model.pt",
    "exp04": EXPERIMENTS_DIR / "exp04_focal_loss_oversampling" / "model.pt",
    "exp05": EXPERIMENTS_DIR / "exp05_focal_loss_sin_dmid" / "model.pt",
    "exp06": EXPERIMENTS_DIR / "exp06_mammoclip_vindr" / "model.pt",
    "exp07": EXPERIMENTS_DIR / "exp07_focal_gamma3_weights_agresivos" / "model.pt",
    "exp08": EXPERIMENTS_DIR / "exp08_ordinal_sord_qwk_descongelado" / "model.pt",
    "exp09": EXPERIMENTS_DIR / "exp09_asymmetric_sord_weighted" / "model.pt",
}

def sha256_of(path, chunk_size=1 << 20):
    ## Hash SHA-256 por bloques, para no cargar el archivo completo en memoria
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

checkpoint_inventory = {}
for name, path in checkpoint_paths.items():
    exists = path.exists()
    entry = {"path": str(path.relative_to(REPO_ROOT)), "exists": bool(exists)}
    if exists:
        entry["size_bytes"] = path.stat().st_size
        entry["sha256"] = sha256_of(path)
    checkpoint_inventory[name] = entry
    print(name, "existe:", entry["exists"], "tamanio:", entry.get("size_bytes"), "sha256:", entry.get("sha256"))

In [ ]:
## Confirmacion puntual pedida en el enunciado: exp08 deberia tener SHA
## con prefijo fc95ab92
exp08_sha = checkpoint_inventory["exp08"].get("sha256", "")
print("sha256 completo de exp08:", exp08_sha)
print("coincide con el prefijo esperado fc95ab92:", exp08_sha.startswith("fc95ab92"))

<h2>Consulta 2a-2b. Arquitectura real de cada checkpoint</h2>
<p>En vez de asumir la arquitectura a partir de las descripciones en texto de cada
experimento, se inspeccionan las claves del <code>state_dict</code> de cada checkpoint.
El unico modelo definido en el codigo actual del repo es
<code>src/models.py:MammoVLM</code> (encoder Mammo-CLIP EfficientNet-B5, con
<code>birads_head</code> y <code>density_head</code>, sin <code>findings_heads</code> y
sin encoder tipo ViT).</p>

In [ ]:
## Deteccion de la arquitectura real de cada checkpoint a partir de las
## claves de su state_dict, sin asumir nada de las descripciones en texto de
## cada experimento
def inspect_state_dict_signature(path):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    sd = ckpt["model_state_dict"]
    keys = list(sd.keys())
    return {
        "n_keys": len(keys),
        "tiene_trunk_vit": any("trunk" in k for k in keys),
        "tiene_findings_heads": any("findings_heads" in k for k in keys),
        "tiene_backbone_efficientnet": any("_blocks" in k or "_conv_stem" in k for k in keys),
    }

for name, entry in checkpoint_inventory.items():
    if not entry["exists"]:
        entry["architecture_signature"] = None
        continue
    sig = inspect_state_dict_signature(checkpoint_paths[name])
    entry["architecture_signature"] = sig
    print(name, sig)

<h2>Consulta 2b-2c. Decision de inclusion o exclusion</h2>
<p>La decision se basa unicamente en lo verificado arriba, no en las hipotesis del
enunciado: existencia fisica del checkpoint y coincidencia de su arquitectura con
<code>MammoVLM</code>, el unico modelo que existe hoy en <code>src/models.py</code>.</p>

In [ ]:
## Decision de inclusion o exclusion basada solo en evidencia verificada:
## existencia fisica del checkpoint y coincidencia arquitectonica con el unico
## modelo que existe hoy en el repo (models.MammoVLM, encoder Mammo-CLIP
## EfficientNet-B5). Esta arquitectura no aparece en git history (un solo commit
## inicial) ni en la carpeta deprecated/.
EXCLUDED = {}
INCLUDED = {}

for name, entry in checkpoint_inventory.items():
    if not entry["exists"]:
        EXCLUDED[name] = "sin checkpoint en disco, model.pt no existe"
        continue
    sig = entry["architecture_signature"]
    if sig["tiene_trunk_vit"] or sig["tiene_findings_heads"]:
        EXCLUDED[name] = (
            "arquitectura ViT tipo BiomedCLIP con findings_heads, no implementada "
            "en el codigo actual del repo. src/models.py solo define MammoVLM con "
            "encoder Mammo-CLIP EfficientNet-B5 y sin findings_heads. Esta "
            "arquitectura previa no existe en git history (un solo commit inicial) "
            "ni en la carpeta deprecated/, por lo que no se puede reconstruir con "
            "modulos reales del repo."
        )
        continue
    if not sig["tiene_backbone_efficientnet"]:
        EXCLUDED[name] = "arquitectura no reconocida, no coincide con MammoVLM del repo"
        continue
    INCLUDED[name] = "arquitectura Mammo-CLIP EfficientNet-B5, coincide con models.MammoVLM"

print("incluidos:")
for k, v in INCLUDED.items():
    print(" ", k, "-", v)
print("excluidos:")
for k, v in EXCLUDED.items():
    print(" ", k, "-", v)

<h2>Consulta 2b. Carga estricta de los checkpoints incluidos</h2>
<p>Para cada experimento incluido se construye <code>MammoVLM</code> con los parametros
de su propio <code>experiment_detail.json</code> (campo <code>config</code>) y se carga
el <code>state_dict</code> con <code>strict=True</code>, usando la ruta de carga propia
del repo (primero se instancia el encoder con el checkpoint preentrenado de Mammo-CLIP,
luego se sobreescribe todo con los pesos finos del experimento).</p>

In [ ]:
## Construccion de la arquitectura correcta por experimento, leyendo su
## propia configuracion desde experiment_detail.json, y carga estricta del
## checkpoint fino sobre esa arquitectura
EXPERIMENT_CONFIG_PATHS = {
    "exp06": EXPERIMENTS_DIR / "exp06_mammoclip_vindr" / "experiment_detail.json",
    "exp07": EXPERIMENTS_DIR / "exp07_focal_gamma3_weights_agresivos" / "experiment_detail.json",
    "exp08": EXPERIMENTS_DIR / "exp08_ordinal_sord_qwk_descongelado" / "experiment_detail.json",
    "exp09": EXPERIMENTS_DIR / "exp09_asymmetric_sord_weighted" / "experiment_detail.json",
}

## Unico backbone usado en todo el repo para estos cuatro experimentos
EFFICIENTNET_NAME = "efficientnet-b5"
MAMMOCLIP_PRETRAINED_PATH = str(REPO_ROOT / "models" / "mammo_clip_b5.tar")

loaded_models = {}
loadability_report = {}

for name in INCLUDED.keys():
    with open(EXPERIMENT_CONFIG_PATHS[name], encoding="utf-8") as f:
        exp_detail = json.load(f)
    exp_cfg = exp_detail["config"]

    model = MammoVLM(
        checkpoint_path=MAMMOCLIP_PRETRAINED_PATH,
        efficientnet_name=EFFICIENTNET_NAME,
        num_birads_classes=exp_cfg.get("num_birads_classes", 5),
        num_density_classes=exp_cfg.get("num_density_classes", 4),
        freeze_encoder=exp_cfg.get("freeze_encoder", True),
        unfreeze_last_n_blocks=exp_cfg.get("unfreeze_last_n_blocks", 0),
    )

    ckpt = torch.load(checkpoint_paths[name], map_location="cpu", weights_only=False)
    loadable = False
    missing, unexpected = None, None
    try:
        missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=True)
        loadable = len(missing) == 0 and len(unexpected) == 0
    except RuntimeError as e:
        print(name, "error de forma al cargar:", str(e)[:200])

    loadability_report[name] = {
        "loadable_strict": bool(loadable),
        "missing_keys": list(missing) if missing is not None else None,
        "unexpected_keys": list(unexpected) if unexpected is not None else None,
    }

    if loadable:
        model.to(DEVICE)
        model.eval()
        loaded_models[name] = model

    print(name, "cargable sin error de forma:", loadable)

<h2>Consulta 3. Correspondencia epoca-checkpoint para exp08</h2>
<p>El checkpoint que se evalua es <code>outputs/experiments/exp08.../model.pt</code>.
Este archivo no trae un campo <code>epoch</code> explicito, pero si trae el historial
completo de entrenamiento (<code>history</code>); su ultima entrada indica a que epoca
corresponden los pesos guardados.</p>
<p>Por separado, <code>outputs/checkpoints/exp08.../</code> guarda checkpoints
intermedios por epoca, pero solo cuando <code>val_loss</code> mejora (criterio de
<code>src/train.py:train_mammovlm</code>). Ese es un archivo distinto al que se evalua
aqui, y puede corresponder a una epoca distinta.</p>

In [ ]:
## Verificacion obligatoria de correspondencia epoca-checkpoint para exp08.
## Se compara la ultima entrada del historial embebido en model.pt (que es el
## checkpoint que vamos a evaluar) contra el val_birads_acc de 0.6921 registrado
## en las notas de experiment_detail.json.
exp08_model_ckpt = torch.load(checkpoint_paths["exp08"], map_location="cpu", weights_only=False)
exp08_history = exp08_model_ckpt["history"]["stage1"]
model_history_last_epoch = exp08_history[-1]

registered_val_acc_rounded = 0.6921
matches_reported = [e for e in exp08_history if round(e["val_birads_acc"], 4) == registered_val_acc_rounded]

print("epoca de la ultima entrada del historial embebido en model.pt:", model_history_last_epoch["epoch"])
print("val_birads_acc de esa epoca:", model_history_last_epoch["val_birads_acc"])
print("epocas cuyo val_birads_acc redondeado es 0.6921:", [e["epoch"] for e in matches_reported])

## Checkpoint de mejor val_loss guardado durante el entrenamiento, en una
## carpeta distinta (outputs/checkpoints/), para contraste
CHECKPOINTS_DIR = REPO_ROOT / "outputs" / "checkpoints"
exp08_best_valloss_path = CHECKPOINTS_DIR / "exp08_ordinal_sord_qwk_descongelado" / "mammovlm_epoch13.pt"
exp08_best_valloss_ckpt = torch.load(exp08_best_valloss_path, map_location="cpu", weights_only=False)

head_key = "birads_head.net.4.weight"
pesos_identicos = torch.equal(
    exp08_model_ckpt["model_state_dict"][head_key],
    exp08_best_valloss_ckpt["model_state_dict"][head_key],
)

epoch_correspondence = {
    "checkpoint_evaluado": str(checkpoint_paths["exp08"].relative_to(REPO_ROOT)),
    "epoca_del_checkpoint_evaluado": model_history_last_epoch["epoch"],
    "val_birads_acc_del_checkpoint_evaluado": model_history_last_epoch["val_birads_acc"],
    "val_birads_acc_registrado_en_notas": registered_val_acc_rounded,
    "coincide_con_lo_registrado": round(model_history_last_epoch["val_birads_acc"], 4) == registered_val_acc_rounded,
    "checkpoint_mejor_val_loss_en_outputs_checkpoints": {
        "ruta": str(exp08_best_valloss_path.relative_to(REPO_ROOT)),
        "epoca": exp08_best_valloss_ckpt["epoch"],
        "val_birads_acc": exp08_best_valloss_ckpt["metrics"]["val_birads_acc"],
        "pesos_de_cabeza_birads_identicos_a_los_evaluados": bool(pesos_identicos),
    },
}
print(json.dumps(epoch_correspondence, indent=2, ensure_ascii=False))

<p>Conclusion de esta verificacion: el checkpoint que se evalua
(<code>outputs/experiments/exp08.../model.pt</code>) corresponde a la epoca final del
entrenamiento (epoca 15), y esa epoca es exactamente la que registra
val_birads_acc=0.6921 en <code>experiment_detail.json</code>. No hay desajuste para el
checkpoint que se usa en este ranking. El checkpoint de mejor val_loss guardado por
separado en <code>outputs/checkpoints/</code> corresponde a la epoca 13
(val_birads_acc=0.6950) y tiene pesos de cabeza BI-RADS distintos: es un archivo
distinto, que aqui no se usa.</p>

<h2>Consulta 4. Sanity check obligatorio sobre exp08</h2>
<p>Se evalua exp08 sobre el val de 2400 y se calcula <code>val_birads_acc</code> (accuracy
simple de la cabeza BI-RADS, la misma metrica que <code>src/train.py:evaluate</code>
calcula durante el entrenamiento). Si no reproduce 0.6921 dentro de tolerancia, el
notebook se detiene aqui y no continua al ranking.</p>

In [ ]:
## Funcion de evaluacion: forward pass sobre el val_loader, sin gradientes,
## devuelve etiquetas verdaderas, predicciones (argmax) y probabilidades (softmax)
## de la cabeza BI-RADS
def evaluate_birads(model, loader, device):
    y_true, y_pred, y_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device)
            targets = batch["birads"].to(device)
            outputs = model(images)
            probs = torch.softmax(outputs["birads"], dim=1)
            preds = torch.argmax(probs, dim=1)
            y_true.append(targets.cpu().numpy())
            y_pred.append(preds.cpu().numpy())
            y_probs.append(probs.cpu().numpy())
    return np.concatenate(y_true), np.concatenate(y_pred), np.concatenate(y_probs)

t0 = time.time()
y_true_exp08, y_pred_exp08, y_probs_exp08 = evaluate_birads(loaded_models["exp08"], val_loader, DEVICE)
elapsed_exp08 = time.time() - t0

reproduced_acc = float(np.mean(y_true_exp08 == y_pred_exp08))
registered_acc = model_history_last_epoch["val_birads_acc"]
abs_diff = abs(reproduced_acc - registered_acc)
TOLERANCE = 0.01
sanity_passed = abs_diff <= TOLERANCE

sanity_check_result = {
    "reproduced_val_birads_acc": reproduced_acc,
    "registered_val_birads_acc": registered_acc,
    "abs_difference": abs_diff,
    "tolerance": TOLERANCE,
    "passed": bool(sanity_passed),
    "eval_time_seconds": elapsed_exp08,
}
print(json.dumps(sanity_check_result, indent=2))

if not sanity_passed:
    raise RuntimeError(
        "el sanity check de exp08 no reproduce el val_birads_acc registrado "
        "dentro de tolerancia, el notebook se detiene aqui, el arnes de "
        "evaluacion no es confiable en este estado"
    )

print("sanity check aprobado, se continua al ranking de los checkpoints sobrevivientes")

<h2>Consulta 5. Evaluacion de los checkpoints sobrevivientes sobre el val de 2400</h2>
<p>Para cada checkpoint incluido se calcula AUC macro one vs rest (5 clases BI-RADS),
accuracy y Quadratic Weighted Kappa, usando <code>src/medical_metrics.py:MedicalMetricsReport</code>
con <code>num_classes=5</code> (la numeracion usada desde exp06 en adelante, indices 0-4
equivalen a BI-RADS 1-5). El preprocesamiento es el mismo para los cuatro, porque los
cuatro comparten la misma clase de dataset y transform.</p>

In [ ]:
## Evaluacion de los demas checkpoints sobrevivientes sobre el mismo val de
## 2400 ya usado para exp08. Las predicciones crudas se guardan en _cache/ para
## auditoria, sin que eso forme parte del resultado final versionado.
remaining = [n for n in loaded_models.keys() if n != "exp08"]

metrics_by_exp = {}

report_exp08 = MedicalMetricsReport(num_classes=5).compute_full_report(y_true_exp08, y_pred_exp08, y_probs_exp08)
mc08 = report_exp08["area_3_medical_metrics"]["multiclass_classification"]
metrics_by_exp["exp08"] = {
    "auc_macro_val": mc08["auc_ovr"]["macro_avg"],
    "accuracy_val": mc08["accuracy"],
    "qwk_val": mc08["quadratic_kappa"],
}
np.savez(CACHE_DIR / "exp08_val_predictions.npz", y_true=y_true_exp08, y_pred=y_pred_exp08, y_probs=y_probs_exp08)
print("exp08", metrics_by_exp["exp08"])

for name in remaining:
    t0 = time.time()
    y_true, y_pred, y_probs = evaluate_birads(loaded_models[name], val_loader, DEVICE)
    elapsed = time.time() - t0

    report = MedicalMetricsReport(num_classes=5).compute_full_report(y_true, y_pred, y_probs)
    mc = report["area_3_medical_metrics"]["multiclass_classification"]
    metrics_by_exp[name] = {
        "auc_macro_val": mc["auc_ovr"]["macro_avg"],
        "accuracy_val": mc["accuracy"],
        "qwk_val": mc["quadratic_kappa"],
    }
    np.savez(CACHE_DIR / f"{name}_val_predictions.npz", y_true=y_true, y_pred=y_pred, y_probs=y_probs)
    print(name, metrics_by_exp[name], "tiempo segundos:", elapsed)

<h2>Consulta 6. Advertencia de comparabilidad</h2>

In [ ]:
## Advertencia de comparabilidad, se deja registrada tanto en el notebook
## como en el JSON de resultados
comparability_warning = (
    "exp01, exp02, exp04 y exp05 se entrenaron sobre un pool multi-dataset "
    "(arquitectura tipo BiomedCLIP con VinDr, CBIS-DDSM, CDD-CESM, DMID e INbreast) y "
    "ademas quedan excluidos de este ranking porque su arquitectura no esta "
    "implementada en el codigo actual del repo. exp03 queda excluido por no tener "
    "checkpoint en disco. Los cuatro experimentos que sobreviven (exp06, exp07, exp08, "
    "exp09) se entrenaron unicamente sobre VinDr-Mammo (VinDr-only), por lo que la "
    "comparacion entre ellos sobre el val de VinDr es una prueba justa y homogenea en "
    "dominio, arquitectura y preprocesamiento. No existe evidencia recuperable en este "
    "repo para comparar estos cuatro contra exp01-05 bajo el mismo criterio de "
    "validacion."
)
print(comparability_warning)

<h2>Consulta 7. Tabla de ranking final</h2>

In [ ]:
## Metadatos de arquitectura, dominio de entrenamiento y preprocesamiento
## por experimento incluido, para dejar constancia explicita en la tabla final
EXP_METADATA = {
    "exp06": {
        "arquitectura": "Mammo-CLIP EfficientNet-B5, encoder totalmente congelado, dual head BIRADS(5)/densidad(4)",
        "dominio_entrenamiento": "VinDr-only",
        "preprocesamiento": "MammoCLIPTransform 1520x912, CLAHE, normalizacion ImageNet, sin augmentacion",
    },
    "exp07": {
        "arquitectura": "Mammo-CLIP EfficientNet-B5, encoder totalmente congelado, dual head BIRADS(5)/densidad(4)",
        "dominio_entrenamiento": "VinDr-only",
        "preprocesamiento": "MammoCLIPTransform 1520x912, CLAHE, normalizacion ImageNet, sin augmentacion",
    },
    "exp08": {
        "arquitectura": "Mammo-CLIP EfficientNet-B5, 2 bloques finales descongelados, dual head BIRADS(5)/densidad(4)",
        "dominio_entrenamiento": "VinDr-only",
        "preprocesamiento": "MammoCLIPTransform 1520x912, CLAHE, normalizacion ImageNet, sin augmentacion",
    },
    "exp09": {
        "arquitectura": "Mammo-CLIP EfficientNet-B5, 2 bloques finales descongelados, dual head BIRADS(5)/densidad(4)",
        "dominio_entrenamiento": "VinDr-only",
        "preprocesamiento": "MammoCLIPTransform 1520x912, CLAHE, normalizacion ImageNet, sin augmentacion",
    },
}

ranking = []
for name, m in metrics_by_exp.items():
    row = {"experimento": name}
    row.update(m)
    row.update(EXP_METADATA[name])
    ranking.append(row)

ranking.sort(key=lambda r: r["auc_macro_val"], reverse=True)

for i, row in enumerate(ranking, start=1):
    print(i, row["experimento"], "AUC_val=%.4f" % row["auc_macro_val"],
          "acc_val=%.4f" % row["accuracy_val"], "QWK_val=%.4f" % row["qwk_val"])

exp08_es_numero_uno = ranking[0]["experimento"] == "exp08"
print("numero uno segun AUC macro de validacion:", ranking[0]["experimento"])
print("exp08 es el numero uno:", exp08_es_numero_uno)

In [ ]:
## Ensamblado y persistencia del resultado definitivo. Este archivo es la
## evidencia reproducible de la seleccion, no las salidas embebidas del notebook.
resultados_finales = {
    "generado_en_utc": datetime.now(timezone.utc).isoformat(),
    "alcance": (
        "seleccion de modelo por AUC macro one vs rest de la cabeza BI-RADS sobre "
        "el conjunto de VALIDACION de VinDr, no sobre el test"
    ),
    "conjunto_validacion": {
        "funcion_split": "src/train.py:split_train_val",
        "funcion_reserva_test": "src/train.py:separate_test_set_vindr",
        "loader_registros": "src/data_loading.py:load_vindr_records",
        "random_seed": split_config.random_seed,
        "val_fraction": split_config.val_fraction,
        "estratificado_por": "birads",
        "n_registros_vindr_totales": n_total_records,
        "n_test_reservado_split_oficial": len(test_df),
        "n_train_val_antes_del_split": len(train_val_df),
        "n_train": len(train_df),
        "n_val": len(val_df),
    },
    "inventario_checkpoints": checkpoint_inventory,
    "incluidos": INCLUDED,
    "excluidos": EXCLUDED,
    "cargabilidad_arquitectura": loadability_report,
    "exp08_correspondencia_epoca_checkpoint": epoch_correspondence,
    "exp08_sanity_check": sanity_check_result,
    "advertencia_comparabilidad": comparability_warning,
    "ranking_val_auc_macro_descendente": ranking,
    "exp08_es_numero_uno": bool(exp08_es_numero_uno),
    "numero_uno": ranking[0]["experimento"],
}

RESULTADOS_PATH = NOTEBOOK_DIR / "resultados_ranking.json"
with open(RESULTADOS_PATH, "w", encoding="utf-8") as f:
    json.dump(resultados_finales, f, indent=2, ensure_ascii=False)

print("resultado guardado en:", RESULTADOS_PATH)

<h2>Bootstrap pareado de la diferencia de AUC: exp08 (condicion C8) contra exp09 (condicion C9)</h2>
<p>
El ranking dio exp09 con AUC macro 0.7578 y exp08 con 0.7561, una diferencia de 0.0017.
Para decidir si esa diferencia es real o ruido no se comparan dos intervalos marginales,
porque ambos modelos predicen sobre las mismas 2400 muestras y sus AUC estan
correlacionados. Se estima el intervalo de confianza de la DIFERENCIA con un bootstrap
pareado: en cada iteracion se remuestrea un unico conjunto de indices y se evalua a los
dos modelos sobre ese mismo remuestreo.
</p>
<p>
El remuestreo es estratificado por clase verdadera. El AUC es macro one-vs-rest sobre
cinco clases BI-RADS con clases raras; un remuestreo simple podria dejar sin muestras a
una clase y volver indefinido su AUC. Estratificando se conservan los conteos por clase
y el promedio macro queda bien definido en cada iteracion. Como el macro no pondera por
prevalencia, fijar los conteos no sesga el estimando.
</p>
<p>
Regla de lectura: si el intervalo de la diferencia contiene el cero, los modelos son
estadisticamente indistinguibles en validacion y aplica el desempate de la regla de
seleccion. (En este notebook, C8/exp08 fue finalmente adoptada como condicion definitiva
y C9/exp09 quedo descartada por colapso en el conjunto de test; ver
docs/condition_mapping.md.)
</p>

In [11]:
import json
import numpy as np
from sklearn.metrics import roc_auc_score
## MedicalMetricsReport y CACHE_DIR ya existen desde las celdas 2 y 1
from pathlib import Path

## Reconstruccion autocontenida de la ruta al cache, identica a la celda 1.
## El notebook vive en experiments/seleccion_val/, y el cache es la subcarpeta _cache
## junto a el. Path.cwd() apunta al directorio desde el que corre el kernel de Jupyter,
## que normalmente es la carpeta del notebook. Si tu kernel arranca en la raiz del repo,
## usa la variante comentada de abajo.
CACHE_DIR = Path.cwd() / "_cache"
## Variante si el kernel arranca en la raiz del repo en lugar de la carpeta del notebook:
## CACHE_DIR = Path("experiments/seleccion_val/_cache")

## Carga de predicciones por muestra desde el cache que escribio la celda 22.
## No se usan variables en memoria: la celda 22 sobrescribe y_true/y_probs en su
## bucle, por lo que exp09 no queda accesible como variable. El cache si lo tiene.
d08 = np.load(CACHE_DIR / "exp08_val_predictions.npz")
d09 = np.load(CACHE_DIR / "exp09_val_predictions.npz")
y_true_a, probs_a = d08["y_true"], d08["y_probs"]   ## exp08
y_true_b, probs_b = d09["y_true"], d09["y_probs"]   ## exp09

## Validez del pareo: mismas muestras, mismo orden. val_loader no baraja, se verifica.
assert y_true_a.shape == y_true_b.shape, "exp08 y exp09 tienen distinto numero de muestras"
assert np.array_equal(y_true_a, y_true_b), "las etiquetas no coinciden muestra a muestra, el pareo seria invalido"
y_true = y_true_a
LABELS = (0, 1, 2, 3, 4)   ## BI-RADS k mapeado a indice k-1
assert set(np.unique(y_true)).issubset(set(LABELS)), "hay etiquetas fuera de 0 a 4"


def auc_repo(yt, yp):
    ## AUC macro one-vs-rest calculado igual que el ranking, con MedicalMetricsReport.
    ## compute_full_report exige y_pred; para el AUC solo importan y_true y y_probs,
    ## pero se pasa el argmax para cumplir la firma.
    yhat = np.argmax(yp, axis=1)
    rep = MedicalMetricsReport(num_classes=5).compute_full_report(yt, yhat, yp)
    return rep["area_3_medical_metrics"]["multiclass_classification"]["auc_ovr"]["macro_avg"]


def auc_fast(yt, yp):
    ## Version sklearn, mucho mas barata para 2000 remuestreos.
    return roc_auc_score(yt, yp, multi_class="ovr", average="macro", labels=list(LABELS))


## Se elige la funcion del bucle: si sklearn coincide con el repo en la muestra
## completa, se usa la rapida; si no, se usa la del repo aunque sea lenta. Asi el
## bootstrap reproduce exactamente el AUC del ranking sin pagar el costo si no hace falta.
a_repo = auc_repo(y_true, probs_a)
a_fast = auc_fast(y_true, probs_a)
if abs(a_repo - a_fast) < 1e-6:
    auc_fn = auc_fast
    print("AUC de sklearn coincide con el del repo (diff < 1e-6), se usa la via rapida")
else:
    auc_fn = auc_repo
    print("AUC de sklearn NO coincide con el del repo (diff=%.6f), se usa MedicalMetricsReport" % abs(a_repo - a_fast))

## Compuerta de correctitud: los puntos deben reproducir el ranking antes de confiar.
auc_a_full = auc_fn(y_true, probs_a)
auc_b_full = auc_fn(y_true, probs_b)
print("AUC exp08 full: %.4f  (esperado 0.7561)" % auc_a_full)
print("AUC exp09 full: %.4f  (esperado 0.7578)" % auc_b_full)


def paired_stratified_bootstrap(y_true, probs_a, probs_b, auc_fn, n_boot=2000, seed=42, labels=LABELS):
    ## Pareado: un unico remuestreo por iteracion, aplicado identico a ambos modelos,
    ## para capturar su correlacion (predicen sobre las mismas imagenes) y medir la
    ## diferencia con el poder correcto.
    ## Estratificado: se remuestrea dentro de cada clase verdadera conservando sus
    ## conteos, para que el AUC one-vs-rest quede definido en cada iteracion.
    rng = np.random.default_rng(seed)
    idx_por_clase = {c: np.where(y_true == c)[0] for c in labels}
    a_boot = np.empty(n_boot)
    b_boot = np.empty(n_boot)
    diff_boot = np.empty(n_boot)
    for i in range(n_boot):
        partes = [rng.choice(idx_por_clase[c], size=len(idx_por_clase[c]), replace=True) for c in labels]
        idx = np.concatenate(partes)
        yt = y_true[idx]
        a_boot[i] = auc_fn(yt, probs_a[idx])
        b_boot[i] = auc_fn(yt, probs_b[idx])
        diff_boot[i] = b_boot[i] - a_boot[i]
    def ci(v):
        ## intervalo percentil al 95 por ciento
        return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))
    return ci(a_boot), ci(b_boot), diff_boot, ci(diff_boot)


ci_a, ci_b, diff_boot, ci_diff = paired_stratified_bootstrap(y_true, probs_a, probs_b, auc_fn, n_boot=2000, seed=42)
p_b_supera_a = float(np.mean(diff_boot > 0))

print("")
print("Intervalos al 95 por ciento (2000 remuestreos, semilla 42):")
print("  exp08 AUC: %.4f  CI [%.4f, %.4f]" % (auc_a_full, ci_a[0], ci_a[1]))
print("  exp09 AUC: %.4f  CI [%.4f, %.4f]" % (auc_b_full, ci_b[0], ci_b[1]))
print("  diferencia exp09 menos exp08: %.4f  CI [%.4f, %.4f]" % (auc_b_full - auc_a_full, ci_diff[0], ci_diff[1]))
print("  P(exp09 supera a exp08): %.3f" % p_b_supera_a)
print("")
if ci_diff[0] <= 0.0 <= ci_diff[1]:
    print("Lectura: el CI de la diferencia contiene el cero. exp08 y exp09 son")
    print("indistinguibles en validacion, aplica el desempate de la regla de seleccion.")
else:
    print("Lectura: el CI de la diferencia NO contiene el cero, hay que revisar la seleccion.")

## Persistencia en _outputs, que tu .gitignore ignora. No commitear aun, hasta decidir
## como se redacta el resultado en la tesis.
res = {
    "auc_exp08_full": auc_a_full,
    "auc_exp09_full": auc_b_full,
    "ci_exp08": ci_a,
    "ci_exp09": ci_b,
    "diff_full": auc_b_full - auc_a_full,
    "ci_diff": ci_diff,
    "p_exp09_supera_exp08": p_b_supera_a,
    "n_boot": 2000,
    "seed": 42,
}
out_dir = CACHE_DIR.parent / "_outputs"
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / "bootstrap_exp08_exp09.json", "w") as f:
    json.dump(res, f, indent=2)
print("resultado guardado en _outputs/bootstrap_exp08_exp09.json")

AUC de sklearn coincide con el del repo (diff < 1e-6), se usa la via rapida
AUC exp08 full: 0.7561  (esperado 0.7561)
AUC exp09 full: 0.7578  (esperado 0.7578)

Intervalos al 95 por ciento (2000 remuestreos, semilla 42):
  exp08 AUC: 0.7561  CI [0.7362, 0.7768]
  exp09 AUC: 0.7578  CI [0.7353, 0.7783]
  diferencia exp09 menos exp08: 0.0017  CI [-0.0086, 0.0108]
  P(exp09 supera a exp08): 0.627

Lectura: el CI de la diferencia contiene el cero. exp08 y exp09 son
indistinguibles en validacion, aplica el desempate de la regla de seleccion.
resultado guardado en _outputs/bootstrap_exp08_exp09.json
